### Roboflow Dataset

#### Download dataset from Roboflow

In [ ]:
import os
import io
import zipfile
import requests

# Roboflow public dataset URL
url = "https://public.roboflow.com/ds/cEbvuuZwUx?key=xJzZQRRk02"

# Base directory (override with AQUARIUM_DATASET_DIR env variable)
base_dir = os.getenv("AQUARIUM_DATASET_DIR", "Yolo_test")

# Create 'dataset' subfolder
target_dir = os.path.join(base_dir, "dataset")
os.makedirs(target_dir, exist_ok=True)

print("Downloading dataset...")
response = requests.get(url, stream=True)
if response.status_code == 200:
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        z.extractall(target_dir)
    print(f"Dataset downloaded and extracted to {target_dir}")
else:
    print("Download failed:", response.status_code, response.text)

#### Modify the dataset from Roboflow

In [ ]:
# Dataset root folder (override with AQUARIUM_DATASET_PATH env variable)
dataset_folder = os.getenv("AQUARIUM_DATASET_PATH", "dataset/roboflow_Aquarium_Combined.v6i.yolov8")

# Original class mapping from data.yaml
original_class_map = {
    0: 'fish',
    1: 'jellyfish',
    2: 'penguin',
    3: 'shark',
    4: 'puffin',
    5: 'stingray',
    6: 'starfish'
}

# Classes to merge into single "fish" class
classes_to_keep_and_merge = ['fish', 'shark', 'stingray']
new_fish_class_id = 0

# Process labels in train, validation, and test sets
for data_split in ['train', 'valid', 'test']:
    
    labels_path = os.path.join(dataset_folder, data_split, 'labels')
    print(f"--- Processing folder: {labels_path} ---")

    label_files = os.listdir(labels_path)

    for file_name in label_files:
        if not file_name.endswith('.txt'):
            continue

        file_path = os.path.join(labels_path, file_name)
        lines_to_keep = []

        with open(file_path, 'r') as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            original_id = int(parts[0])
            class_name = original_class_map[original_id]

            if class_name in classes_to_keep_and_merge:
                new_line = f"{new_fish_class_id} {parts[1]} {parts[2]} {parts[3]} {parts[4]}"
                lines_to_keep.append(new_line)

        with open(file_path, 'w') as f:
            f.write('\n'.join(lines_to_keep))
            
print("\n--- All label files have been modified successfully! ---")

##### Verify label changes

In [ ]:
import os

# Dataset root (override with AQUARIUM_DATASET_PATH env variable)
dataset_folder = os.getenv("AQUARIUM_DATASET_PATH", "dataset/roboflow_Aquarium_Combined.v6i.yolov8")

data_splits = ['train', 'valid', 'test']

print(f"--- Displaying all labels in '{dataset_folder}' ---")

for split in data_splits:
    labels_dir = os.path.join(dataset_folder, split, 'labels')
    
    print(f"\n--- Checking Folder: {labels_dir} ---")

    if not os.path.isdir(labels_dir):
        print("Directory not found.")
        continue

    for filename in os.listdir(labels_dir):
        if filename.endswith('.txt'):
            file_path = os.path.join(labels_dir, filename)
            
            with open(file_path, 'r') as f:
                content = f.read().strip()

            print(f"File: {filename}")
            if content:
                print(content)
            else:
                print("(This file is empty)")

print("\n--- Script Finished ---")